# Capstone, Brief A: RAG over the 10-Ks

**User:** an equity research analyst.

**Goal:** ask questions about the risk factors of 8 companies and get an answer with cited sources (file + chunk id), a confidence score, and a refusal if nothing relevant is found.

Flow:
1. Load the 10-K excerpts.
2. Chunk with `RecursiveCharacterTextSplitter` and keep source metadata.
3. Embed the chunks and store them in Chroma.
4. Dense + BM25 retrieval, combined with RRF, then reranked with a cross-encoder.
5. Generate answers with citations.
6. Guardrail (refusal threshold) and confidence.
7. Evaluate 6 questions: answer, citation correct, hallucination.

In [1]:
import sys
from pathlib import Path

# Find the repo root (the folder with course_utils.py), so the notebook runs from any subfolder
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'course_utils.py').exists())
sys.path.insert(0, str(ROOT))
print('Repo root:', ROOT.name)

Repo root: rag-10k-capstone


## 1. & 2. Load the excerpts and chunk them

First, we load the 8 10-K excerpts. Second, we split them into chunks of 600 **characters** with an overlap of 80.

Why 600 characters: each excerpt is only ~200–290 words, so this gives ~4 chunks per document and keeps the risk sections (e.g. Apple supply chain vs. Apple FX) apart. Much bigger chunks would mean one chunk per document.

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the 8 excerpts and keep file name + company as metadata
DOCS = sorted((ROOT / 'data/sample_10k_excerpts').glob('*.txt'))
docs = [{'source': p.name, 'company': p.stem.split('_')[0], 'text': p.read_text()} for p in DOCS]
print(f'Loaded {len(docs)} docs:', [d['company'] for d in docs])

# Split into chunks; each chunk gets an id like "Apple_10k_excerpt.txt::2" for the citations
splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80)
chunks = []
for d in docs:
    for i, text in enumerate(splitter.split_text(d['text'])):
        chunks.append({'id': f"{d['source']}::{i}", 'text': text,
                       'source': d['source'], 'company': d['company']})

# Print the number of chunks to test
print(f'{len(chunks)} chunks')

Loaded 8 docs: ['Amazon', 'Apple', 'ExxonMobil', 'JPMorgan', 'Microsoft', 'Pfizer', 'Tesla', 'Walmart']
32 chunks


## 3. Embed the chunks and store them

To make the chunks searchable, we turn them into numbers ("vectors") with an embedding model. Chunks with a similar meaning end up close to each other in that vector space. We store the vectors in ChromaDB.

In [3]:
import chromadb
from sentence_transformers import SentenceTransformer

# Small, fast embedding model we used in class
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# In-memory Chroma collection (deleted first, so re-running the cell starts clean)
db = chromadb.EphemeralClient()
try:
    db.delete_collection("ten_k_excerpts")
except Exception:
    pass
col = db.create_collection("ten_k_excerpts")

In [4]:
# Embed all chunks and store them together with their metadata and ids
texts = [c['text'] for c in chunks]
embeddings = embedder.encode(texts, normalize_embeddings=True).tolist()
metadata = [{'company': c['company'], 'source': c['source']} for c in chunks]
ids = [c['id'] for c in chunks]

# Add the chunks to the Chroma collection
col.add(documents=texts, embeddings=embeddings, metadatas=metadata, ids=ids)
print(f"Collection has {col.count()} items.")

Collection has 32 items.


The collection has 32 items, one per chunk. The embedding part is done, so now we build the retrieval.

## 4. Hybrid retrieval (dense + BM25) and reranking

Dense retrieval matches on meaning, BM25 matches on the actual words. Dense can miss exact terms (e.g. "Azure"), BM25 misses paraphrases. Combining both gives us more relevant chunks.

For dense retrieval, we embed the question with the same model as above and ask ChromaDB for the nearest chunks.

I write six small functions:
- `bm25_retrieval`: finds chunks with the same words
- `dense_retrieval`: finds chunks with a similar meaning
- `rrf`: combines two rankings into one list (Reciprocal Rank Fusion)
- `hybrid_retrieval`: runs both retrievers and combines them with `rrf`
- `rerank`: re-scores the candidates with a cross-encoder
- `hybrid_rerank`: puts it all together

In [5]:
import re
from rank_bm25 import BM25Okapi

# Simple tokenizer: lowercase and split into words
def tok(text):
    return re.findall(r'\w+', text.lower())

# BM25 index over all chunks
bm25 = BM25Okapi([tok(c['text']) for c in chunks])

# Return the ids of the k chunks with the highest BM25 score
def bm25_retrieval(question, k=15):
    scores = bm25.get_scores(tok(question))
    top = scores.argsort()[::-1][:k]
    return [chunks[i]['id'] for i in top]

In [6]:
# Return the ids of the k chunks closest to the question in vector space
def dense_retrieval(question, k=15):
    q_embedding = embedder.encode([question], normalize_embeddings=True).tolist()
    results = col.query(query_embeddings=q_embedding, n_results=k)
    return results['ids'][0]

In [7]:
# Reciprocal Rank Fusion: each list adds 1 / (k + rank) to a chunk's score.
# It only uses ranks, so BM25 and dense scores don't need the same scale.
# k=60 is the standard value.
def rrf(rank_lists, k=60, top=15):
    scores = {}
    for rank_list in rank_lists:
        for pos, chunk_id in enumerate(rank_list):
            scores[chunk_id] = scores.get(chunk_id, 0) + 1 / (k + pos + 1)
    return sorted(scores, key=scores.get, reverse=True)[:top]

In [8]:
# Run both retrievers and combine their rankings
def hybrid_retrieval(question, k=15):
    return rrf([bm25_retrieval(question, k), dense_retrieval(question, k)], top=k)

In [9]:
from sentence_transformers import CrossEncoder

# The cross-encoder reads question and chunk together -> a more precise relevance score
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# Look up chunks by their id
by_id = {c['id']: c for c in chunks}

# Score each candidate and keep the best ones, together with their score
def rerank(question, chunk_ids, keep=5):
    scores = reranker.predict([[question, by_id[cid]['text']] for cid in chunk_ids])
    best = sorted(zip(chunk_ids, scores), key=lambda x: x[1], reverse=True)[:keep]
    return [{'chunk_id': cid, 'score': float(score), 'text': by_id[cid]['text'],
             'source': by_id[cid]['source'], 'company': by_id[cid]['company']}
            for cid, score in best]

In [10]:
# Full retrieval: hybrid search for 15 candidates, then rerank and keep the best 5
def hybrid_rerank(question, k=15, keep=5):
    return rerank(question, hybrid_retrieval(question, k), keep=keep)

In [11]:
# Quick check: top 5 hybrid results for three different question types
def show(q, k=5):
    for cid in hybrid_retrieval(q, k=k):
        print(cid, '|', by_id[cid]['text'][:120].replace('\n', ' '))
    print("-------------")

show("What are Apple's main supply chain risks?")   # company + topic
show("Which company mentions Azure?")               # exact term -> BM25
show("Who depends on consumer spending?")           # paraphrase -> dense

Apple_10k_excerpt.txt::2 | Supply Chain and Operational Risks - The Company relies on a small number of outsourcing partners for   manufacturing of
Apple_10k_excerpt.txt::0 | APPLE INC. -- Form 10-K (excerpt, public domain)  Item 1A. Risk Factors  The Company's business, reputation, results of 
Tesla_10k_excerpt.txt::0 | TESLA, INC. -- Form 10-K (excerpt, public domain)  Item 1A. Risk Factors  Manufacturing and Supply Chain We have experie
Apple_10k_excerpt.txt::1 | Macroeconomic and Industry Risks - The Company is subject to global and regional economic and political   conditions, in
Pfizer_10k_excerpt.txt::2 | Clinical Trial and Regulatory Risk Drug development is inherently uncertain. Many compounds that enter clinical trials f
-------------
Amazon_10k_excerpt.txt::0 | AMAZON.COM, INC. -- Form 10-K (excerpt, public domain)  Item 1A. Risk Factors  Intense Competition The online and physic
Apple_10k_excerpt.txt::4 | Legal and Regulatory Compliance - The Company is subject to laws and 

Looks good. Side note: only Amazon mentions Azure (as a competitor), Microsoft's excerpt doesn't, so Amazon on top is correct.

## 5. Generate the answers

`hybrid_rerank` gives us the most relevant chunks. Now we build a message with the question and those chunks and send it to the LLM.

For transparency, every chunk gets a number and a label (file::chunk id + company), and the LLM has to cite the numbers. If the context doesn't contain the answer, the LLM should say "I don't know."

In [12]:
# Rules for the LLM: only use the context, cite [n], don't mix up companies, write full sentences, otherwise "I don't know."
# The "full sentences" rule was added after the first runs: the LLM often just repeated the chunk labels (Q2, Q3).
SYSTEM_PROMPT = (
    "You are a careful assistant for an equity research analyst. "
    "Answer ONLY based on the provided context. "
    "Each context chunk starts with a number and its source, e.g. [1] (source: Apple_10k_excerpt.txt::2 | company: Apple). "
    "Cite every statement with the chunk number in square brackets, e.g. [1] or [2]. "
    "If the question is about a specific company, use ONLY chunks whose company matches that company. "
    "Never attribute information from one company to another. "
    "Write full sentences that name the companies and put the chunk number, e.g. [1], after each statement. "
    "Do not copy the (source: ...) labels into the answer. "
    "If the answer is not in the provided context, say 'I don't know.'"
)

In [13]:
# Every chunk is labelled, e.g. "[1] (source: Apple_10k_excerpt.txt::2 | company: Apple)".
# Without labels the LLM mixed up companies (it gave Tesla's "vehicles" risk to Apple).
# The LLM cites [n], and [n] maps back to the file + chunk id.
def create_message(question, context_chunks):
    context = "\n\n".join(
        f"[{i+1}] (source: {c['chunk_id']} | company: {c['company']})\n{c['text']}"
        for i, c in enumerate(context_chunks))
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}]

## 6. Guardrail and confidence

Every chunk from `hybrid_rerank` has a score from the cross-encoder: positive = relevant, negative = not relevant. I use it twice:

1. **Guardrail.** Chunks below `THRESHOLD` are dropped before the LLM sees them. If no chunk is left, we refuse and don't call the LLM at all.
2. **Confidence.** The best chunk's score is turned into a 0–1 value with a sigmoid: 0 → 0.5, +2.6 → 0.93, −11 → 0.00.

**Why −2:** I started with 0, but that refused the Tesla minerals question (Q2), whose correct chunk only scored −1.37. With −2 it gets answered, while the CEO question (−6.82) and the off-topic question (−10.96) are still refused. Setting it to −100 switches the guardrail off (see Q5 and Q6).

In [14]:
import math

# Chunks with a lower rerank score are treated as irrelevant (see above)
THRESHOLD = -2

# Turn the rerank score into a 0-1 confidence
def to_confidence(score):
    return 1 / (1 + math.exp(-score))

In [15]:
from course_utils import get_llm_client
client, model = get_llm_client()

REFUSAL = "I can't answer this from the 10-K excerpts (no chunk passed the relevance threshold)."

# Full pipeline: retrieve -> rerank -> guardrail -> LLM. Returns a dict for printing and evaluation.
def ask_llm(question, k=3, candidates=5, threshold=None):
    threshold = THRESHOLD if threshold is None else threshold
    reranked = hybrid_rerank(question, keep=candidates)
    # Guardrail: keep only chunks above the threshold, max. k of them
    context = [c for c in reranked if c['score'] > threshold][:k]
    # Nothing relevant -> refuse without calling the LLM
    if not context:
        return {'question': question, 'answer': REFUSAL, 'citations': [],
                'confidence': to_confidence(reranked[0]['score']), 'refused': True}
    out = client.chat.completions.create(
        model=model, messages=create_message(question, context),
        temperature=0.1, max_tokens=512).choices[0].message.content
    return {'question': question, 'answer': out, 'citations': context,
            'confidence': to_confidence(context[0]['score']), 'refused': False}

In [16]:
# Print a result the way the brief asks for: answer, cited excerpts (file + chunk id), confidence
def print_result(r):
    print("QUESTION:  ", r['question'])
    print("ANSWER:    ", r['answer'])
    print("CITATIONS: ")
    if not r['citations']:
        print("   (none - refused)")
    for i, c in enumerate(r['citations']):
        print(f"   [{i+1}] {c['chunk_id']}  (rerank score {c['score']:.2f})")
        print(f"       \"{c['text'][:150].replace(chr(10), ' ')}...\"")
    print(f"CONFIDENCE: {r['confidence']:.2f}")
    print("-" * 80)

In [17]:
# Quick test with the question where the LLM first mixed up Apple and Tesla
print_result(ask_llm("What are Apple's main supply chain risks?"))

QUESTION:   What are Apple's main supply chain risks?
ANSWER:     [1] Apple's main supply chain risks include disruptions in the operations of its outsourcing partners, particularly those located in the Asia-Pacific region, which could be caused by semiconductor shortages, public health issues, geopolitical tension, or natural disasters.

[1] Additionally, Apple's supply chain has experienced periodic constraints in semiconductor and other component supply, which has affected its operations at times.
CITATIONS: 
   [1] Apple_10k_excerpt.txt::2  (rerank score 2.59)
       "Supply Chain and Operational Risks - The Company relies on a small number of outsourcing partners for   manufacturing of many of its products, includi..."
   [2] Tesla_10k_excerpt.txt::0  (rerank score -0.81)
       "TESLA, INC. -- Form 10-K (excerpt, public domain)  Item 1A. Risk Factors  Manufacturing and Supply Chain We have experienced significant supply chain ..."
   [3] Apple_10k_excerpt.txt::0  (rerank score -1

## 7. Evaluation: 5 + 1 test questions

Each question tests a different ability of the pipeline (precise fact, company filter, multi-document, paraphrase, hallucination trap, off-topic). For every question: the expected answer (ground truth from the excerpts), the pipeline output, and my evaluation below it.

Evaluation criteria per question:
- **Answer correct?** (yes / partly / no)
- **Citation correct?** Do the cited `file::chunk` ids contain the answer?
- **Hallucination?** Does the answer contain anything that is not in the cited chunks?

### Q1: How much do generic and biosimilar entrants typically reduce sales of an originator drug after it loses patent exclusivity?
- **Tests:** A precise number from a single document; the company is not named, so retrieval has to find it.
- **Expected:** "70% or more in the year following exclusivity loss" (`Pfizer_10k_excerpt.txt::1`)

In [18]:
print_result(ask_llm("How much do generic and biosimilar entrants typically reduce sales of an originator drug after it loses patent exclusivity?"))

QUESTION:   How much do generic and biosimilar entrants typically reduce sales of an originator drug after it loses patent exclusivity?
ANSWER:     [1] Generic and biosimilar entrants typically reduce sales of an originator product by 70% or more in the year following exclusivity loss.
CITATIONS: 
   [1] Pfizer_10k_excerpt.txt::1  (rerank score 8.06)
       "Patent Expirations and Generic Competition A significant portion of our revenues comes from products that face loss of patent exclusivity in the next ..."
CONFIDENCE: 1.00
--------------------------------------------------------------------------------


**Evaluation Q1**
- Answer correct: Yes, spot on.
- Citation correct: Yes
- Hallucination: None
- Comment: Even after re-running the question, the result was robust and showed the 70%. Confidence 1.00.

### Q2: Which critical minerals does Tesla name as a cost risk?
- **Tests:** A specific company + exact terms; the other 7 companies must be filtered out.
- **Expected:** Lithium, nickel and cobalt (`Tesla_10k_excerpt.txt::1`)

In [19]:
print_result(ask_llm("Which critical minerals does Tesla name as a cost risk?"))

QUESTION:   Which critical minerals does Tesla name as a cost risk?
ANSWER:     [1] Tesla names lithium, nickel, and cobalt as key minerals that have volatile availability and price, posing a cost risk.
CITATIONS: 
   [1] Tesla_10k_excerpt.txt::1  (rerank score -1.37)
       "Battery and Raw Materials The availability and price of key minerals -- including lithium, nickel and cobalt -- has been volatile. Our long-term cost ..."
   [2] Tesla_10k_excerpt.txt::0  (rerank score -1.85)
       "TESLA, INC. -- Form 10-K (excerpt, public domain)  Item 1A. Risk Factors  Manufacturing and Supply Chain We have experienced significant supply chain ..."
CONFIDENCE: 0.20
--------------------------------------------------------------------------------


**Evaluation Q2**
- Answer correct: Yes, though the confidence is quite low (0.20).
- Citation correct: Yes. It also found one additional Tesla chunk.
- Hallucination: None to be found
- Comment: With a threshold of 0, the question got refused: the correct chunk only scored −1.37 and didn't pass our hurdle. After lowering the threshold to −2, it was answered.

### Q3: Which companies say sanctions linked to the Russia-Ukraine conflict affect their business?
- **Tests:** Combining two documents into one answer.
- **Expected:** ExxonMobil and JPMorgan (`ExxonMobil_10k_excerpt.txt::1`, `JPMorgan_10k_excerpt.txt::1`)

In [20]:
print_result(ask_llm("Which companies say sanctions linked to the Russia-Ukraine conflict affect their business?"))

QUESTION:   Which companies say sanctions linked to the Russia-Ukraine conflict affect their business?
ANSWER:     [1] JPMorgan says sanctions, export controls, and counter-measures may affect their cross-border activities due to ongoing tensions involving Russia, the conflict in Ukraine, and competition between the United States and China.

[2] ExxonMobil mentions that sanctions related to the Russia-Ukraine conflict have constrained their operations in certain jurisdictions.
CITATIONS: 
   [1] JPMorgan_10k_excerpt.txt::1  (rerank score 2.96)
       "Geopolitical and Macroeconomic Risk The Firm operates globally and is exposed to geopolitical risk, including ongoing tensions involving Russia, the c..."
   [2] ExxonMobil_10k_excerpt.txt::1  (rerank score 1.09)
       "Commodity Price Volatility Crude oil, natural gas and refined product prices are highly volatile and influenced by geopolitical events, OPEC+ decision..."
CONFIDENCE: 0.95
-------------------------------------------------

**Evaluation Q3**
- Answer correct: Yes, meets the expectation.
- Citation correct: Yes
- Hallucination: None to be found
- Comment: Very high confidence (0.95). This test shows that chunks from two different companies can be retrieved and combined into one answer. In the first runs the LLM only repeated the chunk labels; the "full sentences" rule in the system prompt fixed that.

### Q4: Which retailers warn that tighter household budgets are changing what their customers buy?
- **Tests:** Paraphrase: the excerpts say "consumer spending" / "disposable income", not "household budgets" (dense retrieval should help).
- **Expected:** Amazon and Walmart (`Amazon_10k_excerpt.txt::1`, `Walmart_10k_excerpt.txt::1`)

In [21]:
print_result(ask_llm("Which retailers warn that tighter household budgets are changing what their customers buy?"))

QUESTION:   Which retailers warn that tighter household budgets are changing what their customers buy?
ANSWER:     I can't answer this from the 10-K excerpts (no chunk passed the relevance threshold).
CITATIONS: 
   (none - refused)
CONFIDENCE: 0.00
--------------------------------------------------------------------------------


**Evaluation Q4**
- Answer correct: No, the question got refused.
- Citation correct: n/a
- Hallucination: n/a
- Comment: This question was meant to test semantic closeness, but sadly failed: the reranker scored the right Amazon/Walmart chunks far below the threshold. At least it's the "safe" kind of failure: no answer instead of a wrong one. A stronger reranker or a better embedding model could help here.

### Q5: Who is Tesla's CEO?
- **Tests:** Hallucination trap: the excerpt mentions "our Chief Executive Officer" but never names him. The LLM knows the name from training and must not use it.
- **Expected:** Refuse / "not in the documents"

In [22]:
print_result(ask_llm("Who is Tesla's CEO?"))

QUESTION:   Who is Tesla's CEO?
ANSWER:     I can't answer this from the 10-K excerpts (no chunk passed the relevance threshold).
CITATIONS: 
   (none - refused)
CONFIDENCE: 0.00
--------------------------------------------------------------------------------


In [23]:
# Experiment: guardrail switched off, so the LLM gets the chunks anyway
print_result(ask_llm("Who is Tesla's CEO?", threshold=-100))

QUESTION:   Who is Tesla's CEO?
ANSWER:     I don't know.
CITATIONS: 
   [1] Tesla_10k_excerpt.txt::0  (rerank score -6.82)
       "TESLA, INC. -- Form 10-K (excerpt, public domain)  Item 1A. Risk Factors  Manufacturing and Supply Chain We have experienced significant supply chain ..."
   [2] Tesla_10k_excerpt.txt::3  (rerank score -10.64)
       "Key-Personnel and Concentration Risk Our success depends on the continued service of our Chief Executive Officer and other key personnel. The loss of ..."
   [3] Apple_10k_excerpt.txt::1  (rerank score -11.05)
       "Macroeconomic and Industry Risks - The Company is subject to global and regional economic and political   conditions, including inflation, recessions,..."
CONFIDENCE: 0.00
--------------------------------------------------------------------------------


**Evaluation Q5**
- Answer correct: Yes, expected behaviour.
- Citation correct: n/a
- Hallucination: No, which is exactly what we hoped for.
- Comment: The excerpts never name Elon Musk as Tesla's CEO, so the right answer is not to answer. The threshold already refuses it (best chunk −6.82). With the guardrail switched off (threshold −100, cell above), the LLM sees the chunks, and our system prompt still makes it say "I don't know." So both layers work, and there's no sign of hallucination.

### Q6: What is the capital of France?
- **Tests:** Off-topic question: the guardrail must refuse without calling the LLM.
- **Expected:** Refusal by the threshold guardrail

In [24]:
print_result(ask_llm("What is the capital of France?"))

QUESTION:   What is the capital of France?
ANSWER:     I can't answer this from the 10-K excerpts (no chunk passed the relevance threshold).
CITATIONS: 
   (none - refused)
CONFIDENCE: 0.00
--------------------------------------------------------------------------------


In [25]:
# Experiment: guardrail switched off, so the LLM gets the chunks anyway
print_result(ask_llm("What is the capital of France?", threshold=-100))

QUESTION:   What is the capital of France?
ANSWER:     I don't know.
CITATIONS: 
   [1] JPMorgan_10k_excerpt.txt::1  (rerank score -10.96)
       "Geopolitical and Macroeconomic Risk The Firm operates globally and is exposed to geopolitical risk, including ongoing tensions involving Russia, the c..."
   [2] JPMorgan_10k_excerpt.txt::2  (rerank score -11.04)
       "Regulatory and Litigation Risk Banking is one of the most heavily regulated industries. New rules -- on capital, liquidity, consumer protection, clima..."
   [3] ExxonMobil_10k_excerpt.txt::2  (rerank score -11.19)
       "Operational and Safety Our operations involve significant risks including spills, fires, explosions and natural disasters. Despite extensive safety pr..."
CONFIDENCE: 0.00
--------------------------------------------------------------------------------


**Evaluation Q6**
- Answer correct: Yes, it's no answer.
- Citation correct: n/a
- Hallucination: No, again exactly what we hoped for.
- Comment: The question is clearly off-topic, and the threshold refuses it (best chunk −10.96). Interestingly, with the threshold at −100 (cell above) an answer is generated, but fortunately it's "I don't know." This again shows how relevant the threshold is: it's the first line of defence, the system prompt the second.

### Evaluation summary

| # | Tests | Answer | Citation correct? | Hallucination? | Confidence |
|---|---|---|---|---|---|
| Q1 | Precise fact | ✅ Correct | Yes | No | 1.00 |
| Q2 | Company + exact terms | ✅ Correct (only after threshold −2) | Yes | No | 0.20 |
| Q3 | Two documents | ✅ Correct | Yes | No | 0.95 |
| Q4 | Paraphrase | ❌ Wrongly refused | n/a | No | 0.00 |
| Q5 | Hallucination trap | ✅ Refused | n/a | No | 0.00 |
| Q6 | Off-topic | ✅ Refused | n/a | No | 0.00 |

**Result:** 5 of 6 correct, 1 wrong refusal, and **no hallucinations**.

What went wrong and why:
- **Q4, wrong refusal:** the small reranker (trained on web search, not on finance) doesn't see that "household budgets" means "consumer spending".
- **Threshold trade-off:** 0 was too strict (refused Q2), −100 lets everything through. −2 is a compromise: Q2 is answered, Q5/Q6 are still refused. For an analyst, a wrong refusal costs some time, a hallucination costs trust, so I'd rather stay strict.
- **Answer format:** in the first runs the small 3B model often just repeated the chunk labels (Q2, Q3). One extra rule in the system prompt ("write full sentences…") fixed it.
- **Company mix-up:** with threshold −2, the irrelevant Tesla chunk (−0.81) gets through for the Apple question again, but thanks to the labels and the company rule the LLM only uses the Apple chunks.
- **Confidence** shows how relevant the best chunk is, not whether the LLM's answer is correct.